## Linear orchestrator (PCA / LN / penalized regressions)

**Run order:** **Imports** → **Configuration** → **Load data** → **Model registry** → **Run**.

1. **Imports** — paths, third-party packages, `utils` / `models` (reloads `models.linear`).
2. **Configuration** — dataset, sample, `REVISED_MACRO`, horizons, `RESULTS_CSV` (uses `REPO_ROOT` from Imports).
3. **Load data** — builds `X`, `xr`, `dates`.
4. **Model registry** — `build_model_factories` + `SELECTED_MODELS`.
5. **Run** — tqdm, OOS R², RSZ line, CSV.

**RSZ / `rsz_pvalue`:** `utils.base_utils.RSZ_Signif` (Bianchi et al., 2021 replication style). Compares OOS forecasts to the **expanding historical mean** of the target (`gap` aligns information timing; rows with NaN forecasts are excluded). Builds \(f_t\), fits **OLS of \(f_t\)** on **a constant** with **HAC (`maxlags=12`)**. Returns **`1 − F(df)(t)`** for that intercept (**small ⇒** stronger evidence the model beats the historical-mean benchmark on this test). Not a generic p-value for R² alone.

**Macro timing:** If `REVISED_MACRO` is **False**, `expanding_window(realtime=True)` overwrites the **`fred`** block each OOS date (`ForecastVintageMacroStore`). If **True**, macro stays the static shifted FRED panel in `X`.


In [5]:
# Imports & repo path — run this cell before any other code cell.
import importlib
import os
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

_cur = Path.cwd().resolve()
_repo_root = None
for p in [_cur, *_cur.parents]:
    if (p / "models").is_dir() and (p / "utils").is_dir():
        _repo_root = p
        break
if _repo_root is None:
    raise RuntimeError(
        "Cannot find repo root (need folders models/ and utils/). cwd="
        + str(_cur)
        + ". cd into TIO4900-Replication or start Jupyter from the repo."
    )

REPO_ROOT = str(_repo_root)
REPOROOT = _repo_root

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import utils.base_utils as bu
import utils.window_utils as wu
from utils.macro_grouping import add_group_level, build_full_group_mapping, groups_as_array

import models.linear as _ml  # noqa: E402
importlib.reload(_ml)

from models.base import (
    PCABaselineModel,
    PCAFirst8PCsWithCPModel,
    LudvigsonNgWithCPModel,
)
from models.linear import *

print("Imports ok | REPO_ROOT =", REPO_ROOT)


/home/ulrikts/Documents/NTNU/TIO4900-Replication/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports ok | REPO_ROOT = /home/ulrikts/Documents/NTNU/TIO4900-Replication


In [6]:
# ----------------------------
# Configuration — requires Imports cell (defines REPO_ROOT, pd, np, …)
# ----------------------------
if "REPO_ROOT" not in globals():
    raise RuntimeError("Run the **Imports** cell above first.")

DATASET = "kr"  # lw | kr | gsw
START_DATE = "1971-08-31"
#END_DATE = "2018-12-31"
END_DATE = '2025-06-30'

REVISED_MACRO = False  # True: static shifted FRED-MD in X, expanding_window(realtime=False)
# False: macro in X is only a column template; expanding_window(realtime=True) reloads
#     forecast-vintage macro each OOS date via utils.forecast_vintages.ForecastVintageMacroStore

FREQUENCY = "monthly"  # monthly: monthly yields + get_monthly_forward_rates + h=1 xr | annual: yearly + get_forward_rates

HOLDING_MONTHS_ANNUAL = 12
HOLDING_MONTHS_MONTHLY = 1

GAP = 0  # timing vs realized return in expanding_window / RSZ (use 0 for h=1 monthly as in CT-style stack)
OOS_START = pd.Timestamp("1990-01-31")

TARGET_MATURITIES = ["24", "36", "48", "60", "84", "120"]
CP_COLS = ["24", "36", "48", "60"]

# With FREQUENCY="monthly", get_monthly_forward_rates → 10 cols (12,…,120); FWD_ONLY_N=10 = full curve.
FWD_ONLY_N = 10

MATURITIES_ANNUAL_DEFAULT = [str(i) for i in range(12, 121) if i % 12 == 0]
MATURITIES_MONTHLY_DEFAULT = [str(i) for i in range(1, 121)]

FFILL_PANEL = False

RESULTS_CSV = os.path.join(REPO_ROOT, "notebooks", "orchestrators", "results", "linear_runs.csv")


In [7]:
# Load data (uses bu, wu, FREQUENCY, DATASET, … from Imports + Configuration)
if "bu" not in globals():
    raise RuntimeError("Run **Imports** then **Configuration**, then this cell.")

if FREQUENCY not in {"annual", "monthly"}:
    raise ValueError('FREQUENCY must be "annual" or "monthly".')
if FREQUENCY == "monthly" and DATASET == "gsw":
    raise ValueError('Monthly xr needs lw/kr monthly yields (not gsw).')

if FREQUENCY == "annual":
    yields = bu.get_yields(
        type=DATASET,
        start=START_DATE,
        end=END_DATE,
        maturities=MATURITIES_ANNUAL_DEFAULT,
    )
    forward = bu.get_forward_rates(yields)
    xr = bu.get_excess_returns(yields, horizon=HOLDING_MONTHS_ANNUAL).dropna()
else:
    yields = bu.get_yields(
        type=DATASET,
        start=START_DATE,
        end=END_DATE,
        maturities=MATURITIES_MONTHLY_DEFAULT,
    )
    forward = bu.get_monthly_forward_rates(yields)
    xr = bu.get_excess_returns(yields, horizon=HOLDING_MONTHS_MONTHLY).dropna()

fred_md_start_date = pd.to_datetime(START_DATE) - pd.DateOffset(months=6)
fred_md_raw = bu.get_fred_data(
    "data/2026-01-MD.csv", start=fred_md_start_date, end=END_DATE
)

monthly_yields = bu.get_yields(
    type=DATASET,
    start=START_DATE,
    end=END_DATE,
    maturities=[str(i) for i in range(1, 121)],
)
monthly_xr = bu.get_excess_returns(monthly_yields, horizon=1).dropna()

fred_md = fred_md_raw.shift(1)
fred_md = fred_md.drop(columns=["TWEXAFEGSMTHx", "ACOGNO"])
fred_md = fred_md.loc[START_DATE:END_DATE]

yields = yields.loc[yields.index <= xr.index[-1]]
forward = forward.loc[forward.index <= xr.index[-1]]
xr = xr.loc[xr.index <= xr.index[-1]]
fred_md = fred_md.loc[fred_md.index <= xr.index[-1]]
monthly_xr = monthly_xr.loc[monthly_xr.index <= xr.index[-1]]

s2g = build_full_group_mapping(fred_md, forward, yields)

# fred block uses shifted revised FRED-MD for column layout. If REVISED_MACRO is False,
# expanding_window(realtime=True) replaces these values each date with vintages (see Run cell).
X = pd.concat([fred_md, forward, yields], axis=1, keys=["fred", "forward", "yields"])
X = add_group_level(X, s2g, level_name="group")
X = X.sort_index(axis=1, level="group")
groups = groups_as_array(X, level="group")

if FFILL_PANEL:
    X = X.ffill()

dates = xr.index

for m in TARGET_MATURITIES:
    if m not in xr.columns:
        raise KeyError(f"Maturity {m} not in xr columns: got {list(xr.columns[:5])}...")

if X["forward"].shape[1] < FWD_ONLY_N:
    raise ValueError(
        f"Forward block has {X['forward'].shape[1]} columns; need ≥ FWD_ONLY_N={FWD_ONLY_N}. "
        "Use annual lw or reduce FWD_ONLY_N."
    )

print(
    "dataset:",
    DATASET,
    "| xr:",
    xr.shape,
    "| X:",
    X.shape,
    "| forward cols:",
    X["forward"].shape[1],
    "| FWD_ONLY_N:",
    FWD_ONLY_N,
    "| macro in window:",
    "revised static fred" if REVISED_MACRO else "ForecastVintageMacroStore (realtime=True)",
    "| frequency:",
    FREQUENCY,
)

dataset: kr | xr: (646, 10) | X: (646, 254) | forward cols: 10 | FWD_ONLY_N: 10 | macro in window: ForecastVintageMacroStore (realtime=True) | frequency: monthly


/home/ulrikts/Documents/NTNU/TIO4900-Replication/utils/macro_grouping.py:219: UserWarning: The following series are defined in get_fredmd_grouping() but are not present in the FRED-MD data: ['ACOGNO', 'TWEXAFEGSMTHx']. They may have been dropped or renamed.
  warnings.warn(
/home/ulrikts/Documents/NTNU/TIO4900-Replication/utils/macro_grouping.py:168: UserWarning: 2 entries in series_to_group are not present in the DataFrame columns: ['ACOGNO', 'TWEXAFEGSMTHx']
  warnings.warn(


In [ ]:
if "RegularizedLinearModel" not in globals():
    raise RuntimeError("Run **Imports** cell first (defines model classes).")
if "FWD_ONLY_N" not in globals():
    raise RuntimeError("Run **Configuration** before this cell (defines FWD_ONLY_N).")


def build_model_factories(xr_panel, cp_cols):
    """Return factories that build a fresh estimator per expanding-window template."""
    return {
        "PCA(8)": lambda: PCAFirst8PCsWithCPModel(
            xr_full=xr_panel,
            cp_cols=cp_cols,
            components=8,
            n_cp_forwards=5,
        ),
        "L&N (2009)": lambda: LudvigsonNgWithCPModel(
            xr_full=xr_panel,
            cp_cols=cp_cols,
            n_factors=8,
            n_cp_forwards=5,
        ),
        "Ridge CP + macro (direct)": lambda: RidgeRawMacroCPModel(
            xr_full=xr_panel,
            cp_cols=cp_cols,
            macro_series="fred",
            forward_series="forward",
        ),
        "Lasso CP + macro (direct)": lambda: LassoRawMacroCPModel(
            xr_full=xr_panel,
            cp_cols=cp_cols,
            macro_series="fred",
            forward_series="forward",
        ),
        "ElasticNet CP + macro (direct)": lambda: ElasticNetRawMacroCPModel(
            xr_full=xr_panel,
            cp_cols=cp_cols,
            macro_series="fred",
            forward_series="forward",
        ),
        f"Ridge {FWD_ONLY_N} forwards + macro (direct)": lambda: RidgeRawMacroFwdDirectModel(
            macro_series="fred",
            forward_series="forward",
            forward_max_cols=FWD_ONLY_N,
        ),
        f"Lasso {FWD_ONLY_N} forwards + macro (direct)": lambda: LassoRawMacroFwdDirectModel(
            macro_series="fred",
            forward_series="forward",
            forward_max_cols=FWD_ONLY_N,
        ),
        f"ElasticNet {FWD_ONLY_N} forwards + macro (direct)": lambda: ElasticNetRawMacroFwdDirectModel(
            macro_series="fred",
            forward_series="forward",
            forward_max_cols=FWD_ONLY_N,
        ),
        # Forward rates only (no macro): RegularizedLinearModel, first FWD_ONLY_N cols of X['forward']
        f"ElasticNet fwd-only ({FWD_ONLY_N})": lambda: RegularizedLinearModel(
            model_type="elasticnet",
            include_fred=False,
            forward_max_cols=FWD_ONLY_N,
        ),
        f"Lasso fwd-only ({FWD_ONLY_N})": lambda: RegularizedLinearModel(
            model_type="lasso",
            include_fred=False,
            forward_max_cols=FWD_ONLY_N,
        ),
        f"Ridge fwd-only ({FWD_ONLY_N})": lambda: RegularizedLinearModel(
            model_type="ridge",
            include_fred=False,
            forward_max_cols=FWD_ONLY_N,
        ),
        # PCR on first FWD_ONLY_N forward rates only (no macro): PCABaselineModel in base.py
        f"PCR forward ({FWD_ONLY_N} fwds, 3 PCs)": lambda: PCABaselineModel(
            components=3,
            series="forward",
            max_cols=FWD_ONLY_N,
        ),
    }


SELECTED_MODELS = [
    #f"PCR forward ({FWD_ONLY_N} fwds, 3 PCs)",
    # "PCA(8)",  # excluded per monthly forward-rate run
    "L&N (2009)",
    #f"ElasticNet {FWD_ONLY_N} forwards + macro (direct)",
    #f"ElasticNet fwd-only ({FWD_ONLY_N})",
]


In [9]:
if "build_model_factories" not in globals():
    raise RuntimeError("Run Imports → Configuration → Load → Model registry, then this cell.")

_factories = build_model_factories(xr, CP_COLS)
for k in SELECTED_MODELS:
    if k not in _factories:
        raise KeyError(f"No factory for '{k}'. Available: {sorted(_factories)}")

records = []
task_list = [(name, mat) for name in SELECTED_MODELS for mat in TARGET_MATURITIES]

print(f"\nScheduled {len(task_list)} tasks ({len(SELECTED_MODELS)} models × {len(TARGET_MATURITIES)} maturities)")
print(f"expanding_window(realtime={not REVISED_MACRO})\n")

for name, maturity in tqdm(task_list, desc="model × maturity"):
    print(f"\n>>> {name} | maturity={maturity} ({FREQUENCY})")
    tmpl = _factories[name]()
    y = xr[maturity].values
    y_hat = wu.expanding_window(
        tmpl,
        X,
        y,
        dates,
        OOS_START,
        gap=GAP,
        refit_freq=1,
        coef_callback=None,
        progress=False,
        realtime=not REVISED_MACRO,
    )
    r2 = float(wu.oos_r2(y, y_hat))
    signif = bu.RSZ_Signif(y, y_hat, gap=GAP)

    row = {
        "time_now": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "model": name,
        "maturity": maturity,
        "r2_oos": r2,
        "rsz_pvalue": signif,
        "dataset": DATASET,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "revised_macro_static_X": REVISED_MACRO,
        "expanding_window_realtime": not REVISED_MACRO,
        "frequency": FREQUENCY,
        "gap": GAP,
    }
    records.append(row)

    print(f"    OOS R² = {r2:.6f}")
    print(f"    RSZ significance (vs hist. mean benchmark) = {signif}")

out_df = pd.DataFrame(records)
os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
out_df.to_csv(RESULTS_CSV, index=False)
print(f"\n=== Saved table to {RESULTS_CSV} ===")
print(out_df.to_string(index=False))


Scheduled 24 tasks (4 models × 6 maturities)
expanding_window(realtime=True)



model × maturity:   0%|          | 0/24 [00:00<?, ?it/s]


>>> PCR forward (10 fwds, 3 PCs) | maturity=24 (monthly)


model × maturity:   4%|▍         | 1/24 [01:54<43:56, 114.63s/it]

    OOS R² = -0.035968
    RSZ significance (vs hist. mean benchmark) = 0.7772092350490654

>>> PCR forward (10 fwds, 3 PCs) | maturity=36 (monthly)


...

    OOS R² = -0.001476
    RSZ significance (vs hist. mean benchmark) = 0.36515821069859133

>>> ElasticNet fwd-only (10) | maturity=84 (monthly)


model × maturity:  96%|█████████▌| 23/24 [2:33:27<04:58, 298.89s/it]

    OOS R² = 0.000034
    RSZ significance (vs hist. mean benchmark) = 0.21180382493214345

>>> ElasticNet fwd-only (10) | maturity=120 (monthly)


model × maturity: 100%|██████████| 24/24 [2:35:50<00:00, 389.58s/it]

    OOS R² = -0.019705
    RSZ significance (vs hist. mean benchmark) = 0.7223161827895503

=== Saved table to /home/ulrikts/Documents/NTNU/TIO4900-Replication/notebooks/orchestrators/results/linear_runs.csv ===
           time_now                                   model maturity      r2_oos  rsz_pvalue dataset start_date   end_date  revised_macro_static_X  expanding_window_realtime frequency  gap
2026-05-12 18:30:32            PCR forward (10 fwds, 3 PCs)       24   -0.035968    0.777209      kr 1971-08-31 2025-06-30                   False                       True   monthly    0
2026-05-12 18:32:26            PCR forward (10 fwds, 3 PCs)       36   -0.028147    0.767254      kr 1971-08-31 2025-06-30                   False                       True   monthly    0
2026-05-12 18:34:20            PCR forward (10 fwds, 3 PCs)       48   -0.017287    0.673169      kr 1971-08-31 2025-06-30                   False                       True   monthly    0
2026-05-12 18:36:15            P